In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [ ]:
# Define State
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int 
    sixes: int

    sr: float
    bpb: float
    boundary_percentage: float
    summary: str

In [22]:
# calculate_sr Node Logic
def calculate_sr(state: BatsmanState)-> BatsmanState:
    balls=state["balls"]
    runs=state["runs"]
    sr=(runs/balls)*100
    
    # Instead of returninf full state we will send partial update
    # As sending full state will cause conflict in parallel workflow
    return {"sr": sr}

In [18]:
# calculate_bpb Node Logic
def calculate_bpb(state: BatsmanState)-> BatsmanState:
    bpb= state["balls"]/(state["fours"] + state["sixes"])

    # Instead of returninf full state we will send partial update
    # As sending full state will cause conflict in parallel workflow
    return {"bpb": bpb}

In [19]:
# calculate_boundary_percentage Node Logic
def calculate_boundary_percentage(state: BatsmanState)-> BatsmanState:
    boundary_per=( ((state["fours"]*4) + (state["sixes"]*6)) / state["runs"] ) *100

    # Instead of returninf full state we will send partial update
    # As sending full state will cause conflict in parallel workflow
    return {"boundary_percentage": boundary_per}

In [15]:
# summary Node Logic
def summary(state: BatsmanState)-> BatsmanState:
    summary = f"""
Strike Rate = {state["sr"]} \n
Balls Per Boundary = {state["bpb"]} \n
Boundary Percentage = {state["boundary_percentage"]}"""
    
    # Instead of returninf full state we will send partial update
    # As sending full state will cause conflict in parallel workflow
    return {"summary": summary}

In [23]:
# Define Graph
mygraph=StateGraph(BatsmanState)

# Add Node
mygraph.add_node("calculate_sr", calculate_sr)
mygraph.add_node("calculate_bpb", calculate_bpb)
mygraph.add_node("calculate_boundary_percentage", calculate_boundary_percentage)
mygraph.add_node("summary", summary)

# Define Edges
mygraph.add_edge(START, "calculate_sr")
mygraph.add_edge(START, "calculate_bpb")
mygraph.add_edge(START, "calculate_boundary_percentage")

mygraph.add_edge("calculate_sr", "summary")
mygraph.add_edge("calculate_bpb", "summary")
mygraph.add_edge("calculate_boundary_percentage", "summary")

mygraph.add_edge("summary", END)


workflow = mygraph.compile()

In [24]:
# Execute the Workflow
initial_state= {
    "runs": 100,
    "balls": 50,
    "fours": 6,
    "sixes": 4
}

final_state= workflow.invoke(initial_state)

print(final_state)


{'runs': 100, 'balls': 50, 'fours': 6, 'sixes': 4, 'sr': 200.0, 'bpb': 5.0, 'boundary_percentage': 48.0, 'summary': '\nStrike Rate = 200.0 \n\nBalls Per Boundary = 5.0 \n\nBoundary Percentage = 48.0'}
